### Notebook to tweek YAIB's preprocessing of the data, to fit a SSL setup
- This means not having labels


#### YAIB preprocessing pipeline

raw data -> split data (funciton) -> preprocess data (class)

In [1]:
import os
import copy
import logging

import gin
import json
import hashlib
import pandas as pd
import polars as pl
from pathlib import Path
import pickle
from timeit import default_timer as timer
from sklearn.model_selection import StratifiedKFold, KFold, StratifiedShuffleSplit, ShuffleSplit
from icu_benchmarks.data.preprocessor import Preprocessor, PandasClassificationPreprocessor, PolarsClassificationPreprocessor
from icu_benchmarks.constants import RunMode
from icu_benchmarks.run_utils import check_required_keys
from icu_benchmarks.data.constants import DataSplit as Split, DataSegment as Segment, VarType as Var


In [2]:
from icu_benchmarks.data.split_process_data import *
from icu_benchmarks.cross_validation import execute_repeated_cv  # adjust if path is different
from icu_benchmarks.run import *
import gin

# Path ti configs
#'os.chdir("/work3/s185395/YAIB/")

# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/Regression.gin")


/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


ParsedConfigFileIncludesAndImports(filename='/work3/s185395/YAIB/configs/tasks/Regression.gin', imports=[], includes=[ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Imports.gin', imports=['icu_benchmarks.data.split_process_data', 'icu_benchmarks.data.loader', 'icu_benchmarks.models.wrappers', 'icu_benchmarks.models.dl_models', 'icu_benchmarks.models.ml_models'], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/PredictionTaskVariables.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/CrossValidation.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Dataloader.gin', imports=[], includes=[])])

In [14]:
# Override the outcome scaling range
gin.bind_parameter("base_regression_preprocessor.outcome_min", 0)
gin.bind_parameter("base_regression_preprocessor.outcome_max", 10)

# Call the preprocessing function with the new scale
data = preprocess_data(
    data_dir=Path("demo_data/los/mimic_demo"),
    seed=2222,
    generate_cache=True,
    load_cache=False,
    debug=False,
    use_static = True,
    runmode=RunMode.regression,
)

/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1140: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1145: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1165: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [15]:
print(type(data['train']['FEATURES']))
print(data['train']['FEATURES'].columns)

<class 'polars.dataframe.frame.DataFrame'>
['stay_id', 'time', 'alb', 'alp', 'alt', 'ast', 'be', 'bicar', 'bili', 'bili_dir', 'bnd', 'bun', 'ca', 'cai', 'ck', 'ckmb', 'cl', 'crea', 'crp', 'dbp', 'fgn', 'fio2', 'glu', 'hgb', 'hr', 'inr_pt', 'k', 'lact', 'lymph', 'map', 'mch', 'mchc', 'mcv', 'methb', 'mg', 'na', 'neut', 'o2sat', 'pco2', 'ph', 'phos', 'plt', 'po2', 'ptt', 'resp', 'sbp', 'temp', 'tnt', 'urine', 'wbc', 'MissingIndicator_alb', 'MissingIndicator_alp', 'MissingIndicator_alt', 'MissingIndicator_ast', 'MissingIndicator_be', 'MissingIndicator_bicar', 'MissingIndicator_bili', 'MissingIndicator_bili_dir', 'MissingIndicator_bnd', 'MissingIndicator_bun', 'MissingIndicator_ca', 'MissingIndicator_cai', 'MissingIndicator_ck', 'MissingIndicator_ckmb', 'MissingIndicator_cl', 'MissingIndicator_crea', 'MissingIndicator_crp', 'MissingIndicator_dbp', 'MissingIndicator_fgn', 'MissingIndicator_fio2', 'MissingIndicator_glu', 'MissingIndicator_hgb', 'MissingIndicator_hr', 'MissingIndicator_inr_pt

#### Next step will try to be to build a dataset class that 
1. takes input like the polardataset classes in YAIB
2. computes the output like the MortalityDataset does it for R's repo 
Then we make sure that the raw datastructures can be preprocessed by YAIB's existing tools and that when the preprocessed data is loaded that it can be inputted into R's implementation 

- Maybe be aware of the paired aspect R did in her work. How is class impalance being handled by YAIB? 

This is what Chat says for now, check conversation again when working 

Polars Input
  → group by stay_id
  → sort by time
  → create (T, F) arrays
  → compute masks
  → compute deltas
  → output (data, times, static, label, mask, delta) per patient


## Understanding PredictionPolarsDataset input and output structure

In [ ]:
## Input, after preprocessing 

# Data dict main structure 
data.keys()
dict_keys(['train', 'val', 'test'])

# Within each split 
data['train'].keys()
dict_keys(['OUTCOME', 'FEATURES'])

# Outcome structure
print(type(data['train']['OUTCOME']))
print(data['train']['OUTCOME'].columns)
<class 'polars.dataframe.frame.DataFrame'>
['stay_id', 'time', 'label']
[num_samples x 3]

# Feature structure 
print(type(data['train']['FEATURES']))
print(data['train']['FEATURES'].columns)
<class 'polars.dataframe.frame.DataFrame'>
['stay_id', 'time', 'alb', 'alp', 'alt', 'ast', 'be', 'bicar', 'bili', 'bili_dir', 'bnd', 'bun', 'ca', 'cai', 'ck', 'ckmb', 'cl', 'crea', 'crp', 'dbp', 'fgn', 'fio2', 'glu', 'hgb', 'hr', 'inr_pt', 'k', 'lact', 'lymph', 'map', 'mch', 'mchc', 'mcv', 'methb', 'mg', 'na', 'neut', 'o2sat', 'pco2', 'ph', 'phos', 'plt', 'po2', 'ptt', 'resp', 'sbp', 'temp', 'tnt', 'urine', 'wbc', 'MissingIndicator_alb', 'MissingIndicator_alp', 'MissingIndicator_alt', 'MissingIndicator_ast', 'MissingIndicator_be', 'MissingIndicator_bicar', 'MissingIndicator_bili', 'MissingIndicator_bili_dir', 'MissingIndicator_bnd', 'MissingIndicator_bun', 'MissingIndicator_ca', 'MissingIndicator_cai', 'MissingIndicator_ck', 'MissingIndicator_ckmb', 'MissingIndicator_cl', 'MissingIndicator_crea', 'MissingIndicator_crp', 'MissingIndicator_dbp', 'MissingIndicator_fgn', 'MissingIndicator_fio2', 'MissingIndicator_glu', 'MissingIndicator_hgb', 'MissingIndicator_hr', 'MissingIndicator_inr_pt', 'MissingIndicator_k', 'MissingIndicator_lact', 'MissingIndicator_lymph', 'MissingIndicator_map', 'MissingIndicator_mch', 'MissingIndicator_mchc', 'MissingIndicator_mcv', 'MissingIndicator_methb', 'MissingIndicator_mg', 'MissingIndicator_na', 'MissingIndicator_neut', 'MissingIndicator_o2sat', 'MissingIndicator_pco2', 'MissingIndicator_ph', 'MissingIndicator_phos', 'MissingIndicator_plt', 'MissingIndicator_po2', 'MissingIndicator_ptt', 'MissingIndicator_resp', 'MissingIndicator_sbp', 'MissingIndicator_temp', 'MissingIndicator_tnt', 'MissingIndicator_urine', 'MissingIndicator_wbc']
[num_samples x 98]

In [6]:
data['train']['OUTCOME'].filter(pl.col('stay_id') == 201006)

stay_id,time,label
i64,duration[ms],f64
201006,0ms,1.68
201006,1h,1.68
201006,2h,1.68
201006,3h,1.67
201006,4h,1.66
…,…,…
201006,6d 20h,0.06
201006,6d 21h,0.05
201006,6d 22h,0.04


In [7]:
data['train']['FEATURES'].filter(pl.col('stay_id') == 201006).sort('time')

stay_id,time,alb,alp,alt,ast,be,bicar,bili,bili_dir,bnd,bun,ca,cai,ck,ckmb,cl,crea,crp,dbp,fgn,fio2,glu,hgb,hr,inr_pt,k,lact,lymph,map,mch,mchc,mcv,methb,mg,na,neut,…,MissingIndicator_cai,MissingIndicator_ck,MissingIndicator_ckmb,MissingIndicator_cl,MissingIndicator_crea,MissingIndicator_crp,MissingIndicator_dbp,MissingIndicator_fgn,MissingIndicator_fio2,MissingIndicator_glu,MissingIndicator_hgb,MissingIndicator_hr,MissingIndicator_inr_pt,MissingIndicator_k,MissingIndicator_lact,MissingIndicator_lymph,MissingIndicator_map,MissingIndicator_mch,MissingIndicator_mchc,MissingIndicator_mcv,MissingIndicator_methb,MissingIndicator_mg,MissingIndicator_na,MissingIndicator_neut,MissingIndicator_o2sat,MissingIndicator_pco2,MissingIndicator_ph,MissingIndicator_phos,MissingIndicator_plt,MissingIndicator_po2,MissingIndicator_ptt,MissingIndicator_resp,MissingIndicator_sbp,MissingIndicator_temp,MissingIndicator_tnt,MissingIndicator_urine,MissingIndicator_wbc
i64,duration[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
201006,0ms,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.516011,0.0,0.0,0.454634,-0.930323,0.681749,-0.580126,-0.658271,0.0,-0.61205,0.422837,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,false,false,true,false,true,true,false,false,false,false,false,true,false,false,false,false,false,true,false,false,false,false,true,true,false,false,true,false,false,false,true,true,true,false
201006,1h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.619331,0.0,0.0,0.454634,-0.930323,0.925322,-0.580126,-0.658271,0.0,-0.61205,0.735582,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,true,true,true,false,true,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,true,true,false,true
201006,2h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.791532,0.0,0.0,0.454634,-0.930323,0.925322,-0.580126,-0.658271,0.0,-0.61205,0.90813,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,true,true,true,false,true,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,true,true,false,true
201006,3h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.860412,0.0,0.0,0.454634,-0.930323,0.844131,-0.580126,-0.658271,0.0,-0.61205,1.199306,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,true,true,true,false,true,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,false,true,true,true
201006,4h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.17161,0.0,0.0,0.454634,-0.930323,0.140477,-0.580126,-0.658271,0.0,-0.61205,0.034602,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,true,true,true,false,true,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,true,true,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
201006,6d 20h,-0.60284,0.0,0.0,0.0,-1.290317,0.221696,0.0,0.0,-0.041957,1.679445,1.03133,2.202911,0.0,0.0,-0.69228,1.108013,0.0,1.067052,0.0,2.424732,-0.639334,0.761649,1.331277,-0.580126,2.467983,-0.758467,-1.038713,0.68166,

## Mortality dataset data output

In [8]:
from icu_benchmarks.data.loader import PredictionPolarsDataset
from torch.utils.data import DataLoader

dataset_class = PredictionPolarsDataset

batch_size=1

train_dataset = dataset_class(data, split=Split.train, ram_cache=False)
train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
    )

In [9]:
# Get one batch from the DataLoader
for batch_idx, (data, labels, pad_mask) in enumerate(train_loader):
    # Print the data and other details of the batch
    print(f"Batch {batch_idx + 1}:")
    print("Data (tensor):", data)
    print("Labels (tensor):", labels)
    print("Padding Mask (tensor):", pad_mask)
    
    # Optionally, inspect the shape of the batch tensors
    print(f"Data shape: {data.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Padding Mask shape: {pad_mask.shape}")
    
    # Break after the first batch to just inspect one batch
    break


Batch 1:
Data (tensor): tensor([[[-0.6028,  0.0000,  0.0000,  ...,  1.0000,  1.0000,  0.0000],
         [-0.6028,  0.0000,  0.0000,  ...,  1.0000,  0.0000,  1.0000],
         [-0.6028,  0.0000,  0.0000,  ...,  1.0000,  0.0000,  1.0000],
         ...,
         [-0.6028,  0.0000,  0.0000,  ...,  1.0000,  1.0000,  1.0000],
         [-0.6028,  0.0000,  0.0000,  ...,  1.0000,  1.0000,  1.0000],
         [-0.6028,  0.0000,  0.0000,  ...,  1.0000,  1.0000,  1.0000]]])
Labels (tensor): tensor([[1.6800, 1.6800, 1.6800, 1.6700, 1.6600, 1.6500, 1.6400, 1.6300, 1.6200,
         1.6100, 1.6000, 1.5900, 1.5800, 1.5700, 1.5600, 1.5500, 1.5400, 1.5300,
         1.5200, 1.5100, 1.5000, 1.4900, 1.4800, 1.4700, 1.4600, 1.4500, 1.4400,
         1.4300, 1.4200, 1.4100, 1.4000, 1.3900, 1.3800, 1.3700, 1.3600, 1.3500,
         1.3400, 1.3300, 1.3200, 1.3100, 1.3000, 1.2900, 1.2800, 1.2700, 1.2600,
         1.2500, 1.2400, 1.2300, 1.2200, 1.2100, 1.2000, 1.1900, 1.1800, 1.1700,
         1.1600, 1.1500, 1.1400

In [11]:
data.shape

torch.Size([1, 169, 96])

In [ ]:
# I need to create all of these 6 from the preprocessed data format above 

Dynamic Features 
[batch_size, num_sensors, num_timesteps]

Static Features
[batch_size, num_static_features]

label_array
[batch_size, 1]

Sensor Mask
[batch_size, num_sensors, num_timesteps]

Time Features
[batch_size, num_timesteps]

Delta Features
[batch_size, num_sensors, num_timesteps]

In [ ]:
# First version of dataset class
# does not handle missing values propperly by using missingness columns 
# Be aware of padding in batch, and also missingness indicators. Are they distinguished between? 
    # Check how R does it in her dataset class, to me it looks like she just combines them so that 
    # ... padding is just the same type of missingness as missing values? 


import torch
import numpy as np
import polars as pl
from torch import Tensor
from typing import Tuple
from torch.utils.data import Dataset


class SSLDataset(CommonPolarsDataset):
    def __init__(self, data: Dict[str, pl.DataFrame], split: str = "train", max_length=2881):
        """
        Initialize the dataset class and load the data.

        Arguments:
            data: A dictionary of Polars DataFrames with keys "dynamic", "static", and "outcome".
            split: The data split (train/val/test).
            max_length: Maximum sequence length for padding.
        """
        # Initialize the base class (CommonPolarsDataset)
        super().__init__(data, split=split, vars=vars)

        # Store max_length for padding
        self.max_length = max_length

    def __getitem__(self, idx: int) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor, Tensor, Tensor]:
        """
        Function to sample from the data split of choice. Used for deep learning implementations.

        Args:
            idx: A specific row index to sample.

        Returns:
            A sample from the data, consisting of data, labels, padding mask, and other arrays.
        """
        # Extracting the stay_id for the specific index
        stay_id = self.outcome_df[self.vars["GROUP"]].unique()[idx]

        # Filter the data to get the relevant data for this stay_id
        window = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(pl.exclude(self.vars["GROUP"])).to_numpy()
        labels = self.outcome_df.filter(pl.col(self.vars["GROUP"]) == stay_id)[self.vars["LABEL"]].to_numpy().astype(float)

        # Handling the case where only one label exists
        if len(labels) == 1:
            # only one label per stay, align with window
            labels = np.concatenate([np.empty(window.shape[0] - 1) * np.nan, labels], axis=0)

        # Padding the data to match the max length
        length_diff = self.max_length - window.shape[0]
        pad_mask = np.ones(window.shape[0])  # Initially, set all to 1

        if length_diff > 0:
            # If the window is shorter than max_length, pad it with zeros
            window = np.concatenate([window, np.ones((length_diff, window.shape[1])) * 0.0], axis=0)
            labels = np.concatenate([labels, np.ones(length_diff) * 0.0], axis=0)
            pad_mask = np.concatenate([pad_mask, np.zeros(length_diff)], axis=0)

        # Handling missing labels
        not_labeled = np.argwhere(np.isnan(labels))
        if len(not_labeled) > 0:
            labels[not_labeled] = -1  # Mark missing labels as -1
            pad_mask[not_labeled] = 0  # Corresponding mask as 0

        # Convert all to the appropriate types
        pad_mask = pad_mask.astype(bool)
        labels = labels.astype(np.float32)
        data = window.astype(np.float32)

        # Additional arrays needed for the restructured data
        times = window[:, 0]  # Assuming the first column is the time
        static = np.zeros_like(window)  # Assuming static features are zero, you should adapt this based on your dataset
        delta = np.diff(times, prepend=0)  # Time difference between readings

        # Return all of the required arrays as tensors
        return (
            torch.from_numpy(data),  # Data tensor
            torch.from_numpy(labels),  # Label tensor
            torch.from_numpy(pad_mask),  # Pad mask tensor
            torch.from_numpy(times),  # Time tensor (needs adjustment based on actual data)
            torch.from_numpy(static),  # Static tensor (needs actual static features)
            torch.from_numpy(delta)   # Delta tensor (time difference)
        )

    def __len__(self) -> int:
        """
        Return the total number of samples in the dataset.
        """
        return len(self.outcome_df[self.vars["GROUP"]])

    def to_tensor(self):
        """
        Convert the data to tensors. This could be used to return the entire dataset in tensor format.
        """
        return [
            self.data_array,
            self.sensor_mask_array,
            self.times_array,
            self.static_array,
            self.label_array,
            self.delta_array,
        ]


In [ ]:
# Second version, includes observation and forecasting + masks 
# doesnot handle static, delta, original mask yet 

import torch
import numpy as np
import polars as pl
from torch.utils.data import Dataset
from typing import Dict, Tuple


class CustomImputationDataset(Dataset):
    def __init__(self, data: Dict[str, pl.DataFrame], split: str = "train", max_length=2881, max_obs=2):
        """
        Initialize the dataset class and load the data.

        Arguments:
            data: A dictionary of Polars DataFrames with keys "dynamic", "static", and "outcome".
            split: The data split (train/val/test).
            max_length: Maximum sequence length for padding.
            max_obs: Number of time bins for the observation window (set to 2 for 2-hour windows).
        """
        super().__init__()

        # Initialize data and variables
        self.data = data
        self.split = split
        self.max_length = max_length
        self.max_obs = max_obs
        self.features_df = data['dynamic']
        self.missingness_mask_df = data['missingness_mask']  # Assuming you have the missingness mask
        self.outcome_df = data['outcome']
        self.vars = {"GROUP": "stay_id", "SEQUENCE": "time", "DYNAMIC": ['hr', 'map', 'sbp', 'dbp', 'resp', 'o2sat']}
    
    def select_observation_forecasting_windows(self, window: np.ndarray, missingness_mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        Method to randomly select the observation window (t0 to t1) and the corresponding forecasting window (t2),
        along with the corresponding observation and forecasting masks.

        Args:
            window: The data for the current stay_id (time series).
            missingness_mask: The missingness mask for the data (1 for observed, 0 for missing).

        Returns:
            A tuple (obs_window, forecast_window, obs_mask, forecast_mask), where all are numpy arrays.
        """
        valid_t2_found = False
        forecast_window = None
        obs_mask = None
        forecast_mask = None

        # Try sampling t1 until we find a valid t2
        while not valid_t2_found:
            # Randomly select t1 (forecasting time) from the available timestamps
            t1_ix = np.random.choice(len(window))  # t1 can be any index within the available time steps

            # Define t0 as the start of the observation window, ensuring it doesn't go below 0
            t0_ix = max(0, t1_ix - self.max_obs)

            # Slice the observation window
            obs_window = window[t0_ix:t1_ix]  # Observation window (from t0 to t1)
            obs_mask = missingness_mask[t0_ix:t1_ix]  # Corresponding observation mask (same length as obs_window)

            # Search for the first time step after t1 with at least one observed value in that time step
            for t2_ix in range(t1_ix, len(window)):
                # Check if at least one value in this time step is observed (based on missingness mask)
                if np.any(missingness_mask[t2_ix] == 1):  # 1 means observed, 0 means missing
                    forecast_window = window[t2_ix:t2_ix + 1]  # Forecast window (just one time step after t1)
                    forecast_mask = missingness_mask[t2_ix:t2_ix + 1]  # Forecasting mask (same length as forecast_window)
                    valid_t2_found = True  # Mark that we found a valid forecasting window
                    break  # Exit the loop once a valid t2 is found

            if not valid_t2_found:
                # If no valid t2 is found, randomly sample another t1 and try again
                continue

        return obs_window, forecast_window, obs_mask, forecast_mask

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Function to sample from the data split of choice. Used for deep learning implementations.

        Args:
            idx: A specific row index to sample.

        Returns:
            A sample from the data, consisting of data, labels, padding mask, and other arrays.
        """
        stay_id = self.outcome_df[self.vars["GROUP"]].unique()[idx]

        # Filter data based on stay_id

        
        window = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(pl.exclude(self.vars["GROUP"])).to_numpy()
        missingness_mask = self.missingness_mask_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(pl.exclude(self.vars["GROUP"])).to_numpy()


        # Extract times array in minues 
        times = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(pl.col('time')).to_numpy()
        times = np.array([x.total_seconds() / 60 for x in times.flatten()])

        # Get the observation and forecasting windows and their corresponding masks
        obs_window, forecast_window, obs_mask, forecast_mask = self.select_observation_forecasting_windows(window, missingness_mask)

        # Padding (if needed, ensure all windows are padded to max_length)
        length_diff = self.max_length - obs_window.shape[0]
        if length_diff > 0:
            obs_window = np.concatenate([obs_window, np.zeros((length_diff, obs_window.shape[1]))], axis=0)
            obs_mask = np.concatenate([obs_mask, np.zeros((length_diff, obs_mask.shape[1]))], axis=0)

        # Create a pad mask where 1 means observed, 0 means padded
        pad_mask = np.ones(obs_window.shape[0])  # Initially, set all to 1
        if length_diff > 0:
            pad_mask = np.concatenate([pad_mask, np.zeros(length_diff)], axis=0)

        # Convert to tensors
        obs_window = torch.from_numpy(obs_window).float()
        forecast_window = torch.from_numpy(forecast_window).float()
        obs_mask = torch.from_numpy(obs_mask).bool()
        forecast_mask = torch.from_numpy(forecast_mask).bool()
        pad_mask = torch.from_numpy(pad_mask).bool()

        # Assuming the 'times' and 'static' arrays are either provided or can be generated.
        times = np.arange(len(obs_window))  # Example times, adjust as needed
        static = np.zeros_like(obs_window)  # Assuming static features are zero, you should adapt this based on your dataset
        delta = np.diff(times, prepend=0)  # Time difference between readings

        # Return all of the required arrays as tensors
        return (
            obs_window,  # Observation window
            forecast_window,  # Forecast window
            obs_mask,  # Observation mask
            forecast_mask,  # Forecasting mask
            torch.from_numpy(times),  # Time tensor (needs adjustment based on actual data)
            torch.from_numpy(static),  # Static tensor (needs actual static features)
            torch.from_numpy(delta)   # Delta tensor (time difference)
        )

    def __len__(self) -> int:
        """
        Return the total number of samples in the dataset.
        """
        return len(self.outcome_df[self.vars["GROUP"]])


#### Next step will be to try and deduct the static, dynamic and the masks for dynamic array in the new dataset function 

remember to check how R handles missing static value, how does she impute them? then you should do the same 
Try and understand how the commonpolarsdataset class is workig
- are they doing some grouping and sorting?? maybe you should do the same
- How are they hadling the stay_id and the time column? i think maybe they are grouping and sorting based on this
- Continue in your chat BaseModule Class Breakdown 

### Below is the actual format of some preprocessed data that ran through their pipeline

In [39]:
import polars as pl
import os

# Path where your Parquet files are saved
folder_path = "/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data_test"

# List all the Parquet files in the folder
parquet_files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]

# Load and inspect each Parquet file
for parquet_file in parquet_files:
    file_path = os.path.join(folder_path, parquet_file)
    
    # Load the Parquet file into a Polars DataFrame
    df = pl.read_parquet(file_path)
    
    # Print the file name and inspect the first few rows of the DataFrame
    print(f"Inspecting {parquet_file}:")
    print(df.head())  # Shows the first few rows
    print(df.shape)  # Shows the number of rows and columns
    print(df.columns)  # Shows the column names
    print("\n")  # Add an empty line for readability


Inspecting train_FEATURES.parquet:
shape: (5, 106)
┌─────────┬────────────┬───────────┬──────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ stay_id ┆ time       ┆ alb       ┆ alp      ┆ … ┆ MissingInd ┆ MissingIn ┆ MissingIn ┆ MissingIn │
│ ---     ┆ ---        ┆ ---       ┆ ---      ┆   ┆ icator_age ┆ dicator_s ┆ dicator_h ┆ dicator_w │
│ i64     ┆ duration[m ┆ f64       ┆ f64      ┆   ┆ ---        ┆ ex        ┆ eight     ┆ eight     │
│         ┆ s]         ┆           ┆          ┆   ┆ bool       ┆ ---       ┆ ---       ┆ ---       │
│         ┆            ┆           ┆          ┆   ┆            ┆ bool      ┆ bool      ┆ bool      │
╞═════════╪════════════╪═══════════╪══════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 223870  ┆ 2h         ┆ -1.048117 ┆ 1.319224 ┆ … ┆ false      ┆ false     ┆ false     ┆ false     │
│ 281609  ┆ 1d         ┆ 0.0       ┆ 0.0      ┆ … ┆ false      ┆ false     ┆ false     ┆ false     │
│ 283875  ┆ 2h         ┆ 0.0       ┆ 0.0

In [51]:
import polars as pl
import os

# Path where your Parquet files are saved
folder_path = "/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data_test"

# List all the Parquet files in the folder
#parquet_file = 'train_FEATURES.parquet'
parquet_file = 'train_OUTCOME.parquet'

file_path = os.path.join(folder_path, parquet_file)

# Load the Parquet file into a Polars DataFrame
df = pl.read_parquet(file_path)
print(df.head())  # Shows the first few rows



shape: (5, 3)
┌─────────┬──────────────┬──────────┐
│ stay_id ┆ time         ┆ label    │
│ ---     ┆ ---          ┆ ---      │
│ i64     ┆ duration[ms] ┆ f64      │
╞═════════╪══════════════╪══════════╡
│ 200001  ┆ 0ms          ┆ 0.324444 │
│ 200001  ┆ 1h           ┆ 0.32     │
│ 200001  ┆ 2h           ┆ 0.315556 │
│ 200001  ┆ 3h           ┆ 0.311111 │
│ 200001  ┆ 4h           ┆ 0.306667 │
└─────────┴──────────────┴──────────┘
